<a href="https://colab.research.google.com/github/waelrash1/ai-for-business-analytics/blob/main/notebooks/Block%2003/Block3_Lab4_Deploy_the_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deploy the model: from a notebook to an app, with Python and an AI partner

**MI6261 AI for Business Analytics, Block 3.** Dr Wael Rashwan, School of Business, Maynooth University. MSc in Business Analytics.

In Lab 3 you built a model that says, when the seller hands a parcel to the carrier, how likely it is to arrive late. A model in a notebook helps nobody. Customer care needs a tool: type in an order, get the risk, the action and a plain explanation. Today you build that tool and test it before anyone uses it.

In each task you:

1. **Plan**: read the question and decide what one result you need.
2. **Prompt**: ask Gemini or Claude for **one small code cell**.
3. **Read** the code before you run it: does the app use the saved model and the threshold you chose, and only features known at handover?
4. **Run** it.
5. **Verify** one result with a second method.
6. **Explain** the result in your own words, for the customer care manager.

The prompt help fades: tasks 1 and 2 give you the prompt and I show them on screen; task 3 gives you the prompt; tasks 4 to 6 you write yourself.

**Rules.** The app uses the model saved to a file, not the one in memory. The threshold comes from costs, not from 0.5. Nothing is shared until it passes tests with known answers. The app says what it must not be used for.

**Data.** 5,000 real delivered orders from the Olist Brazilian E-Commerce Public Dataset (Kaggle, CC BY-NC-SA 4.0), cleaned with the rules of the Block 2 cleaning lab. An order is late when it arrived after the date promised to the customer.

- **Orders**: https://raw.githubusercontent.com/waelrash1/ai-for-business-analytics/refs/heads/main/olist_model_orders.csv

**AI use.** This lab is vibe coding with checks: you may use any AI to write code. You must read, check and explain every cell. Keep a note of what you had to correct in the prompt log at the end.

---
## Task 1: Refit the late-delivery model and save it to a file

**Question.** Can we rebuild the Lab 3 model in one cell and save it so an app can load it?

**Plan.** One cell that produces: `n_train`, `auc_test`.

**A weak prompt:** *"Save my model."* Which model, trained on what, with which features? An app needs the whole pipeline (preparation and model) in one file, trained only on the training orders.

In [ ]:
import pandas as pd
orders_url = "https://raw.githubusercontent.com/waelrash1/ai-for-business-analytics/refs/heads/main/olist_model_orders.csv"

**The prompt.** I use this one on screen. Copy it into Gemini or Claude.

```text
Context: Colab. `orders_url`: 5,000 cleaned, delivered orders. Late = days_late above 0. Known at handover: handling_days, promised_days, distance_km, weight_g, freight, n_items, purchase_month, customer_state, seller_state.
Task: Load `orders`, add late. Split 75/25, stratified. Pipeline: scale the numbers, one-hot the two states, LogisticRegression. Fit on train. Save with joblib as late_model.joblib, load it back and score the test set.
Constraints: random_state=42, stratify; OneHotEncoder(handle_unknown='infrequent_if_exist', min_frequency=20); max_iter=1000; never delivery_days, days_late or review_score.
Format: One cell; print the test AUC.
Check: Store `n_train`, `auc_test` (3 decimals) and `p_late` (test probabilities from the reloaded model).
```

**Paste the AI's code below.** Before you run it, check that it:
- The whole pipeline is saved, not only the logistic regression
- Fitted on X_train only
- No delivery_days, days_late or review_score among the features

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the reloaded model gives exactly the same probabilities as the one in memory

In [ ]:
print(bool(np.allclose(p_late, p_memory)))   # the saved file is the same model

**Explain.** Why save the preparation steps and the model in one file? What would go wrong if the app scaled new orders with its own mean and standard deviation?

In [ ]:
my_explanation = """

"""

---
## Task 2: Choose the warning threshold from costs

**Question.** Above what risk should customer care warn the customer?

**Plan.** One cell that produces: `best_t`, `best_cost`.

**A weak prompt:** *"Use 0.5 as the threshold."* 0.5 ignores what each error costs. A missed late parcel (a likely bad review) costs 5; a warning for a parcel that arrives on time costs 1.

**The prompt.** I use this one on screen. Copy it into Gemini or Claude.

```text
Context: `p_late` and `y_test` from task 1.
Task: For thresholds 0.05 to 0.95 in steps of 0.05: cost = 5 x late parcels not warned + 1 x warnings for on-time parcels. Find the cheapest.
Constraints: No refitting; thresholds rounded to 2 decimals.
Format: One cell; print the best threshold, its cost and the cost of warning nobody.
Check: Store `best_t` and `best_cost`.
```

**Paste the AI's code below.** Before you run it, check that it:
- 5 for a miss, 1 for a needless warning, as asked
- The same p_late; no refitting
- The cost of warning nobody is printed for comparison

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the chosen threshold costs less than warning nobody

In [ ]:
print(best_cost < 5 * int(y_test.sum()))   # cheaper than warning nobody

**Explain.** The theory value is 1 / (1 + 5), about 0.17. Is the threshold you found close to it? Who in the business should confirm the two costs before the app goes live?

In [ ]:
my_explanation = """

"""

---
## Task 3: One function: an order in, the risk and the action out

**Question.** What exactly will the app do with one new order?

**Plan.** One cell that produces: `example_risk`, `example_action`.

**A weak prompt:** *"Make a function that predicts."* Predicts what, from which inputs, and what does the user do with it? The function should take one order and return the risk and the action at the threshold from task 2.

**The prompt, written for you.** Copy it into Gemini or Claude.

```text
Context: `model` (loaded from late_model.joblib), `features`, `best_t`.
Task: `predict(order)`: order is a dict with the nine features. Return a dict with risk (3 decimals) and action: 'Warn the customer' if risk is at least best_t, else 'No action'. Run it on the example order below.
Constraints: Use the loaded model; keep the columns in the order of features.
Format: One cell; print the result for the example.
Check: Store `example_risk` and `example_action`.
```

**Paste the AI's code below.** Before you run it, check that it:
- The loaded model, not a new fit
- The threshold is best_t, not 0.5
- The columns are put in the training order

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: predict() gives the same risk as the pipeline in memory for the same order

In [ ]:
print(round(float(pipe.predict_proba(pd.DataFrame([example])[features])[0, 1]), 3) == example_risk)   # same risk from the pipeline in memory

**Explain.** Tell a customer care agent, in one sentence, what this risk means and what they should do.

In [ ]:
my_explanation = """

"""

---
## Task 4: Test the function with cases whose answers you know

**Question.** Would you notice if the app were broken?

**Plan.** One cell that produces: `n_tests_passed`.

**A weak prompt:** *"Test the function."* Test it against what? Write cases where you know the right answer before you run them, and let the code stop if one fails.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `predict()` and `example` from task 3.
- **Task:** Three assert tests: (1) the example's risk is between 0 and 1; (2) a seller who took 30 days to hand over an order promised in 10 days gets 'Warn the customer'; (3) a 1-day handover, 30 days promised, 10 km, 500 g, freight 10, one item, June, SP to SP gets 'No action'. Count the tests passed.
- **Constraints:** assert statements; each test changes only what it says.
- **Format:** One cell; print the count.
- **Check:** Store `n_tests_passed`.

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- Each expected answer is written before the test runs
- A failing test stops the cell with a message
- The slow case changes only the two day columns

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: all three tests pass

In [ ]:
print(n_tests_passed == 3)   # all three tests pass

**Explain.** Write one more test you would add before customer care uses the app, and say what it protects against.

In [ ]:
my_explanation = """

"""

---
## Task 5: The app and its model card

**Question.** Can a customer care agent use the model without opening a notebook, and know its limits?

**Plan.** One cell that produces: `n_inputs`, `n_card_fields`.

**A weak prompt:** *"Make a Gradio app for my model."* With which inputs, showing what, and with which warning? Ask for the inputs, the outputs and a short model card shown in the app.

**Write your prompt.** Use the five headings. Hints:

- **Context:** `predict()`, `features`, `best_t`, `auc_test`; the states in `orders`.
- **Task:** A dict `card` with five fields: purpose, data, measure, limits, owner. A gr.Interface `app`: seven number inputs and two state dropdowns, outputs the risk and the action, with the card as its description.
- **Constraints:** Build the app; do not launch it in this cell.
- **Format:** One cell.
- **Check:** Store `n_inputs` and `n_card_fields`.

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- The app calls predict(), so it uses the saved model and best_t
- The card says what the app must not be used for
- No launch() in this cell

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the app gives the same action as predict() for the example order

In [ ]:
print(app_predict(*example.values())[1] == example_action)   # the app agrees with predict()

**Explain.** Launch the app (next step below) and try the slow and the easy orders from task 4. Which line of the model card would you show first to a new agent, and why?

In [ ]:
my_explanation = """

"""

---
## Task 6: An LLM explains the prediction, within rules you set

**Question.** Can the app explain a prediction in plain words without inventing anything?

**Plan.** One cell that produces: `n_quoted`, `n_not_given`.

**A weak prompt:** *"Ask the AI to explain the prediction."* The LLM will explain fluently whatever you give it, and may add a delivery date or a number of its own. Give it only the facts and the rules, then check every number in its answer.

**Run this cell first.** It is given to you, not written by the AI.

In [ ]:
# Given to you, not written by the AI: the LLM helper for this lab.
# With a Gemini key in Colab Secrets (name: GEMINI_API_KEY) it calls Gemini.
# Without a key it returns a saved draft written for this lab (for the example order only),
# so the lab runs with no key and no cost.
SAVED_DRAFT = """The model gives this order a risk of 0.455 of arriving after the promised date.
That is above the warning level of 0.2, which the business chose from the cost of each kind of error, so the advice is to warn the customer.
The seller took 4 days to hand the parcel to the carrier, the customer was promised 22 days, and the parcel travels about 1100 km.
This is an estimate, not a delivery date."""

def ask_llm(prompt):
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
    except Exception:
        key = None
    if not key:
        return SAVED_DRAFT
    from google import genai
    client = genai.Client(api_key=key)
    return client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text

**Write your prompt.** Use the five headings. Hints:

- **Context:** `predict()`, `example`, `best_t`. The LLM helper `ask_llm(prompt)` is given in the cell below.
- **Task:** `explain(order)`: build facts_text (the order's values, the risk, the action, the threshold) and a prompt with four rules: use only these facts; never promise a delivery date; mention no person or address; say so if unsure. Call ask_llm. Then list every number in the answer that is not in facts_text.
- **Constraints:** Remove commas inside numbers before comparing.
- **Format:** One cell; print the explanation and the check.
- **Check:** Store `n_quoted` (numbers in the explanation) and `n_not_given` (numbers not in the facts).

In [ ]:
prompt = """
Context:
Task:
Constraints:
Format:
Check:
"""

**Paste the AI's code below.** Before you run it, check that it:
- The rules are in the prompt, not only in your head
- Every number in the answer is checked against the facts
- No personal data goes into the prompt

In [ ]:
# Paste the AI's code here. Read it before you run it.

**Check with a second method.** Run this check: it should print True.

- Second check: the explanation quotes the same risk that predict() returns

In [ ]:
print(str(example_risk) in text)   # the explanation quotes the risk predict() gave

**Explain.** The check finds invented numbers. What can it not find? Give one way the explanation could still mislead an agent, and how you would catch it.

In [ ]:
my_explanation = """

"""

---
## Share your app

Run this in a new cell. Colab prints a public link that works for 72 hours; ask your partner to try the slow and the easy orders from task 4.

```python
app.launch(share=True)
```

**Stretch: a permanent link on Hugging Face Spaces (free account).** Create a new Space with the Gradio SDK. Upload `late_model.joblib` and two files: `app.py` (load the model with joblib, then the code of tasks 3 and 5, ending with `app.launch()`) and `requirements.txt` (one line each: `pandas`, `scikit-learn`, `joblib`, `gradio`). Use the same scikit-learn version as Colab (`import sklearn; print(sklearn.__version__)`), or the saved model may not load. Put no keys and no personal data in a public Space. To use the explanation box in a Space, store the key as a Space secret, never in the code.

## Your deployment note (about 200 words)

**What the app does.** The input, the risk, the action, and the threshold with the costs behind it.

**How you tested it.** Your three tests and the one you would add.

**Limits.** What the model card says it must not be used for, and what could change after launch (new carriers, a strike, a new region).

**The explanation box.** What the number check catches and what it cannot.

**AI use.** Which AI you used, one prompt that worked well, and one thing in the AI's code you had to correct.

---
## Your prompt log

Vibe coding is only safe when you check. Fill in three rows before you leave.

| Task | The prompt that worked (short) | What the AI got wrong, or could have | How you caught it |
|---|---|---|---|
| | | | |
| | | | |
| | | | |